# Sentiment Benchmark

Notebook này benchmark sentiment classification trên **dataset** (`data_cleaned_full.csv`), sau khi chỉ clean text.

Logic chính:
- Train supervised models trên `data_cleaned_full.csv` với nhãn AI ban đầu: `is_relevant`, `manual_label`.
- Test cố định trên `human_gold_500_cleaned_final.csv`, chỉ các dòng human relevant.
- Loại toàn bộ dòng gold khỏi train bằng key mạnh để tránh leakage.
- Benchmark:
  1. TF-IDF + Logistic Regression
  2. PhoBERT-base-v2 fine-tuned
  3. ViSoBERT fine-tuned
  4. Qwen2.5 3B Instruct LLM zero-shot
  5. Qwen2.5 3B Instruct LLM few-shot


In [1]:
# ============================================================
# 0. SETUP + LOAD DATA + PREPARE TRAIN/TEST
# ============================================================

import os
import re
import gc
import time
import json
import shutil
import random
import requests
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# -----------------------------
# Path linh hoạt
# -----------------------------

def find_existing_path(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Không tìm thấy file trong các path: {candidates}")

FULL_PATH = find_existing_path([
    "data_cleaned_full.csv",
    "data_cleaned_full(1).csv",
    "/mnt/data/data_cleaned_full.csv",
    "/mnt/data/data_cleaned_full(1).csv",
])

GOLD_PATH = find_existing_path([
    "human_gold_500_cleaned_final.csv",
    "/mnt/data/human_gold_500_cleaned_final.csv",
])

df = pd.read_csv(FULL_PATH)
gold = pd.read_csv(GOLD_PATH)

print("Full before-calibration dataset:", FULL_PATH)
print("Full shape:", df.shape)
print("Gold set:", GOLD_PATH)
print("Gold shape:", gold.shape)

print("\nFull columns:")
print(df.columns.tolist())

print("\nGold columns:")
print(gold.columns.tolist())


# -----------------------------
# Validate columns
# -----------------------------

required_full_cols = [
    "topic", "text", "clean_text",
    "is_relevant", "manual_label",
    "video_id", "created_at"
]

required_gold_cols = [
    "topic", "text", "clean_text",
    "final_is_relevant", "final_manual_label",
    "video_id", "created_at"
]

missing_full = [c for c in required_full_cols if c not in df.columns]
missing_gold = [c for c in required_gold_cols if c not in gold.columns]

if missing_full:
    raise ValueError(f"Full dataset thiếu cột: {missing_full}")

if missing_gold:
    raise ValueError(f"Gold set thiếu cột: {missing_gold}")


# -----------------------------
# Basic label checks
# -----------------------------

df["is_relevant"] = pd.to_numeric(df["is_relevant"], errors="coerce").fillna(0).astype(int)
df["manual_label"] = pd.to_numeric(df["manual_label"], errors="coerce").fillna(2).astype(int)

gold["final_is_relevant"] = pd.to_numeric(gold["final_is_relevant"], errors="coerce").fillna(0).astype(int)
gold["final_manual_label"] = pd.to_numeric(gold["final_manual_label"], errors="coerce").fillna(2).astype(int)

# Theo quy ước: irrelevant => neutral
df.loc[df["is_relevant"] == 0, "manual_label"] = 2
gold.loc[gold["final_is_relevant"] == 0, "final_manual_label"] = 2

print("\n========== BEFORE-CALIBRATION DATA CHECK ==========")
print("\nAI is_relevant:")
print(df["is_relevant"].value_counts().sort_index())

print("\nAI manual_label:")
print(df["manual_label"].value_counts().sort_index())

invalid_full = ((df["is_relevant"] == 0) & (df["manual_label"] != 2)).sum()
print("\nInvalid full is_relevant = 0 nhưng manual_label != 2:", invalid_full)

print("\nGold final_is_relevant:")
print(gold["final_is_relevant"].value_counts().sort_index())

print("\nGold final_manual_label:")
print(gold["final_manual_label"].value_counts().sort_index())

print("\nGold topic distribution:")
print(gold["topic"].value_counts().sort_index())


# -----------------------------
# Text + leakage key
# -----------------------------

def safe_model_text(row):
    clean_text = row.get("clean_text", "")
    text = row.get("text", "")

    if pd.notna(clean_text) and str(clean_text).strip() != "":
        return str(clean_text)

    if pd.notna(text) and str(text).strip() != "":
        return str(text)

    return ""

def normalize_key_value(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).lower().strip())

def make_leakage_key(frame):
    # Dùng key mạnh để loại gold khỏi train:
    # topic + video_id + created_at + text
    return frame.apply(
        lambda r: "||".join([
            normalize_key_value(r.get("topic", "")),
            normalize_key_value(r.get("video_id", "")),
            normalize_key_value(r.get("created_at", "")),
            normalize_key_value(r.get("text", "")),
        ]),
        axis=1
    )

df["model_text"] = df.apply(safe_model_text, axis=1)
gold["model_text"] = gold.apply(safe_model_text, axis=1)

df["leakage_key"] = make_leakage_key(df)
gold["leakage_key"] = make_leakage_key(gold)

gold_keys_all = set(gold["leakage_key"])

matched_full_gold_rows = df["leakage_key"].isin(gold_keys_all).sum()
print("\nFull rows matched with gold keys:", matched_full_gold_rows)

# -----------------------------
# Sentiment train/test
# -----------------------------

train_df = df[
    (df["is_relevant"] == 1)
    & (~df["leakage_key"].isin(gold_keys_all))
].copy()

# Test: human gold relevant comments
test_df = gold[
    gold["final_is_relevant"] == 1
].copy()

train_df = train_df[train_df["model_text"].str.strip() != ""].copy()
test_df = test_df[test_df["model_text"].str.strip() != ""].copy()

X_train = train_df["model_text"]
y_train = train_df["manual_label"].astype(int)

X_test = test_df["model_text"]
y_test = test_df["final_manual_label"].astype(int)

print("\n========== SENTIMENT TRAIN/TEST BEFORE CALIBRATION ==========")
print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("\nTrain label distribution:")
print(y_train.value_counts().sort_index())

print("\nTest label distribution:")
print(y_test.value_counts().sort_index())

print("\nTrain topic distribution:")
print(train_df["topic"].value_counts().sort_index())

print("\nTest topic distribution:")
print(test_df["topic"].value_counts().sort_index())

overlap = set(train_df["leakage_key"]) & set(test_df["leakage_key"])
print("\nLeakage overlap train/test:", len(overlap))

if len(overlap) > 0:
    raise RuntimeError("Vẫn còn overlap train/test. Cần kiểm tra lại leakage key.")

benchmark_results = []


c:\Users\nguye\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Full before-calibration dataset: data_cleaned_full.csv
Full shape: (10718, 14)
Gold set: human_gold_500_cleaned_final.csv
Gold shape: (500, 12)

Full columns:
['id', 'video_id', 'topic', 'text', 'clean_text', 'created_at', 'source', 'source_item_id', 'like_count', 'reply_count', 'manual_label', 'predicted_label', 'sentiment_score_raw', 'is_relevant']

Gold columns:
['id', 'video_id', 'topic', 'text', 'clean_text', 'created_at', 'source', 'source_item_id', 'like_count', 'reply_count', 'final_is_relevant', 'final_manual_label']

========== BEFORE-CALIBRATION DATA CHECK ==========

AI is_relevant:
is_relevant
0    2393
1    8325
Name: count, dtype: int64

AI manual_label:
manual_label
0    2313
1    2563
2    5842
Name: count, dtype: int64

Invalid full is_relevant = 0 nhưng manual_label != 2: 0

Gold final_is_relevant:
final_is_relevant
0    219
1    281
Name: count, dtype: int64

Gold final_manual_label:
final_manual_label
0    

In [2]:
# ============================================================
# 1. COMMON EVALUATION HELPERS
# ============================================================

id2label = {
    0: "positive",
    1: "negative",
    2: "neutral"
}

label2id = {
    "positive": 0,
    "negative": 1,
    "neutral": 2
}

def evaluate_predictions(model_name, y_true, y_pred, train_size=0, test_size=None, extra_info=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"========== {model_name} ==========")
    print("Accuracy:", round(acc, 4))
    print("Macro-F1:", round(macro_f1, 4))
    print("Weighted-F1:", round(weighted_f1, 4))
    print("Cohen's Kappa:", round(kappa, 4))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            target_names=["positive", "negative", "neutral"],
            digits=4
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(
        cm,
        index=["Human positive", "Human negative", "Human neutral"],
        columns=["Pred positive", "Pred negative", "Pred neutral"]
    )

    print("\nConfusion matrix:")
    print(cm_df)

    result = {
        "model": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "cohen_kappa": kappa,
        "train_size": train_size,
        "test_size": len(y_true) if test_size is None else test_size,
    }

    if extra_info:
        result.update(extra_info)

    benchmark_results.append(result)

    return result, cm_df


class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = ["" if pd.isna(t) else str(t) for t in texts]
        self.labels = [int(x) for x in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        encoding["labels"] = self.labels[idx]
        return encoding


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "cohen_kappa": cohen_kappa_score(labels, preds)
    }


def cleanup_model_objects():
    for var_name in [
        "model",
        "trainer",
        "tokenizer",
        "train_dataset",
        "val_dataset",
        "test_dataset",
        "data_collator"
    ]:
        if var_name in globals():
            del globals()[var_name]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def make_training_args(output_dir, batch_size=8, grad_accum=2, epochs=10, lr=2e-5):
    # Không overwrite_output_dir vì một số version transformers của bạn không nhận tham số này.
    return TrainingArguments(
        output_dir=output_dir,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,

        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,

        weight_decay=0.01,
        warmup_ratio=0.1,

        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,

        fp16=torch.cuda.is_available(),

        save_total_limit=2,
        report_to="none",

        seed=SEED,
        data_seed=SEED,

        dataloader_num_workers=0
    )


def train_transformer_model(
    model_name,
    display_name,
    output_dir,
    use_fast_tokenizer=False,
    max_length=128,
    batch_size=8,
    grad_accum=2,
    epochs=10,
    lr=2e-5
):
    print(f"\n\n========== START TRAINING: {display_name} ==========")

    cleanup_model_objects()
    set_seed(SEED)

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        X_train.tolist(),
        y_train.tolist(),
        test_size=0.1,
        random_state=SEED,
        stratify=y_train
    )

    test_texts = X_test.tolist()
    test_labels = y_test.tolist()

    print("Train split:", len(train_texts))
    print("Validation split:", len(val_texts))
    print("Gold test:", len(test_texts))

    print("\nTrain split label distribution:")
    print(pd.Series(train_labels).value_counts().sort_index())

    print("\nValidation label distribution:")
    print(pd.Series(val_labels).value_counts().sort_index())

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=use_fast_tokenizer
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    model.to(device)

    train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_length=max_length)
    val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, max_length=max_length)
    test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, max_length=max_length)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = make_training_args(
        output_dir=output_dir,
        batch_size=batch_size,
        grad_accum=grad_accum,
        epochs=epochs,
        lr=lr
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2,
                early_stopping_threshold=0.001
            )
        ]
    )

    trainer.train()

    print("\nTraining done.")
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best validation macro-F1:", trainer.state.best_metric)

    log_df = pd.DataFrame(trainer.state.log_history)
    eval_logs = log_df[log_df["eval_macro_f1"].notna()].copy()

    cols = [
        "epoch",
        "eval_loss",
        "eval_accuracy",
        "eval_macro_f1",
        "eval_weighted_f1",
        "eval_cohen_kappa"
    ]

    eval_logs = eval_logs[[c for c in cols if c in eval_logs.columns]]

    print("\nValidation logs:")
    display(eval_logs)

    test_output = trainer.predict(test_dataset)
    y_pred = np.argmax(test_output.predictions, axis=1)
    y_true = np.array(test_labels)

    result, cm_df = evaluate_predictions(
        display_name,
        y_true,
        y_pred,
        train_size=len(train_df),
        test_size=len(test_df),
        extra_info={
            "best_validation_macro_f1": trainer.state.best_metric,
            "best_checkpoint": trainer.state.best_model_checkpoint,
        }
    )

    pred_df = test_df.copy()
    safe_name = display_name.lower().replace(" ", "_").replace("+", "plus").replace("/", "_")
    pred_df[f"pred_{safe_name}"] = y_pred
    pred_df[f"pred_{safe_name}_label_name"] = pd.Series(y_pred).map(id2label).values

    pred_path = f"before_calibration_predictions_{safe_name}.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print("\nSaved predictions:", pred_path)

    return result


In [3]:
# ============================================================
# 2. MODEL 1 — TF-IDF + LOGISTIC REGRESSION
# ============================================================

tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=SEED
        )
    )
])

tfidf_lr.fit(X_train, y_train)
y_pred_tfidf = tfidf_lr.predict(X_test)

evaluate_predictions(
    "TF-IDF + Logistic Regression",
    y_test,
    y_pred_tfidf,
    train_size=len(train_df),
    test_size=len(test_df)
)

pred_df = test_df.copy()
pred_df["pred_tfidf_lr"] = y_pred_tfidf
pred_df["pred_tfidf_lr_label_name"] = pd.Series(y_pred_tfidf).map(id2label).values
pred_df.to_csv("before_calibration_predictions_tfidf_lr.csv", index=False, encoding="utf-8-sig")

print("Saved predictions: before_calibration_predictions_tfidf_lr.csv")


========== TF-IDF + Logistic Regression ==========
Accuracy: 0.6512
Macro-F1: 0.654
Weighted-F1: 0.6504
Cohen's Kappa: 0.4746

Classification report:
              precision    recall  f1-score   support

    positive     0.6977    0.7407    0.7186        81
    negative     0.6442    0.6321    0.6381       106
     neutral     0.6154    0.5957    0.6054        94

    accuracy                         0.6512       281
   macro avg     0.6524    0.6562    0.6540       281
weighted avg     0.6500    0.6512    0.6504       281


Confusion matrix:
                Pred positive  Pred negative  Pred neutral
Human positive             60             11            10
Human negative             14             67            25
Human neutral              12             26            56
Saved predictions: before_calibration_predictions_tfidf_lr.csv


In [4]:
# ============================================================
# 3. MODEL 2 — PHOBERT-BASE-V2 FINE-TUNED
# ============================================================

phobert_result = train_transformer_model(
    model_name="vinai/phobert-base-v2",
    display_name="PhoBERT-base-v2 Fine-tuned",
    output_dir="before_calibration_phobert_sentiment_model",
    use_fast_tokenizer=False,
    max_length=128,
    batch_size=8,
    grad_accum=2,
    epochs=10,
    lr=2e-5
)




========== START TRAINING: PhoBERT-base-v2 Fine-tuned ==========
Train split: 7132
Validation split: 793
Gold test: 281

Train split label distribution:
0    1975
1    2210
2    2947
Name: count, dtype: int64

Validation label distribution:
0    220
1    246
2    327
Name: count, dtype: int64


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 44833.31it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Cohen Kappa
1,1.680550,0.773772,0.674653,0.665501,0.670687,0.493565
2,1.488892,0.716175,0.682219,0.678463,0.681234,0.513342
3,1.066573,0.773338,0.696091,0.686577,0.689396,0.527660
4,0.867815,0.792568,0.709962,0.705388,0.708258,0.550129
5,0.618108,0.885542,0.698613,0.695388,0.697915,0.536689
6,0.365242,1.241483,0.706179,0.703606,0.705639,0.550133


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]



Training done.
Best checkpoint: before_calibration_phobert_sentiment_model\checkpoint-1784
Best validation macro-F1: 0.7053883064613614

Validation logs:


,epoch,eval_loss,eval_accuracy,eval_macro_f1,eval_weighted_f1,eval_cohen_kappa
8,1.0,0.773772,0.674653,0.665501,0.670687,0.493565
18,2.0,0.716175,0.682219,0.678463,0.681234,0.513342
28,3.0,0.773338,0.696091,0.686577,0.689396,0.527660
38,4.0,0.792568,0.709962,0.705388,0.708258,0.550129
48,5.0,0.885542,0.698613,0.695388,0.697915,0.536689
58,6.0,1.241483,0.706179,0.703606,0.705639,0.550133


========== PhoBERT-base-v2 Fine-tuned ==========
Accuracy: 0.7046
Macro-F1: 0.7098
Weighted-F1: 0.7063
Cohen's Kappa: 0.5546

Classification report:
              precision    recall  f1-score   support

    positive     0.8082    0.7284    0.7662        81
    negative     0.7340    0.6509    0.6900       106
     neutral     0.6140    0.7447    0.6731        94

    accuracy                         0.7046       281
   macro avg     0.7188    0.7080    0.7098       281
weighted avg     0.7153    0.7046    0.7063       281


Confusion matrix:
                Pred positive  Pred negative  Pred neutral
Human positive             59              8            14
Human negative              7             69            30
Human neutral               7             17            70

Saved predictions: before_calibration_predictions_phobert-base-v2_fine-tuned.csv


In [5]:
# ============================================================
# 4. MODEL 3 — VISOBERT FINE-TUNED
# ============================================================

visobert_result = train_transformer_model(
    model_name="uitnlp/visobert",
    display_name="ViSoBERT Fine-tuned",
    output_dir="before_calibration_visobert_sentiment_model",
    use_fast_tokenizer=False,
    max_length=128,
    batch_size=8,
    grad_accum=2,
    epochs=10,
    lr=2e-5
)




========== START TRAINING: ViSoBERT Fine-tuned ==========
Train split: 7132
Validation split: 793
Gold test: 281

Train split label distribution:
0    1975
1    2210
2    2947
Name: count, dtype: int64

Validation label distribution:
0    220
1    246
2    327
Name: count, dtype: int64


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 30081.47it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Cohen Kappa
1,1.664972,0.861425,0.595208,0.586014,0.592610,0.373277
2,1.547170,0.880229,0.626734,0.619622,0.625434,0.426469
3,0.944988,0.965104,0.630517,0.622233,0.626544,0.435414
4,0.556226,1.333024,0.621690,0.614671,0.619416,0.416279
5,0.303242,2.000202,0.626734,0.625788,0.627898,0.435697
6,0.189366,2.465451,0.630517,0.626640,0.630594,0.435425
7,0.093842,2.717317,0.630517,0.620379,0.626329,0.426914


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.16it/s]



Training done.
Best checkpoint: before_calibration_visobert_sentiment_model\checkpoint-2676
Best validation macro-F1: 0.6266404120311476

Validation logs:


,epoch,eval_loss,eval_accuracy,eval_macro_f1,eval_weighted_f1,eval_cohen_kappa
8,1.0,0.861425,0.595208,0.586014,0.592610,0.373277
18,2.0,0.880229,0.626734,0.619622,0.625434,0.426469
28,3.0,0.965104,0.630517,0.622233,0.626544,0.435414
38,4.0,1.333024,0.621690,0.614671,0.619416,0.416279
48,5.0,2.000202,0.626734,0.625788,0.627898,0.435697
58,6.0,2.465451,0.630517,0.626640,0.630594,0.435425
68,7.0,2.717317,0.630517,0.620379,0.626329,0.426914


========== ViSoBERT Fine-tuned ==========
Accuracy: 0.6014
Macro-F1: 0.6029
Weighted-F1: 0.6015
Cohen's Kappa: 0.4001

Classification report:
              precision    recall  f1-score   support

    positive     0.6220    0.6296    0.6258        81
    negative     0.6250    0.5660    0.5941       106
     neutral     0.5631    0.6170    0.5888        94

    accuracy                         0.6014       281
   macro avg     0.6034    0.6042    0.6029       281
weighted avg     0.6034    0.6014    0.6015       281


Confusion matrix:
                Pred positive  Pred negative  Pred neutral
Human positive             51             11            19
Human negative             20             60            26
Human neutral              11             25            58

Saved predictions: before_calibration_predictions_visobert_fine-tuned.csv


In [6]:
# ============================================================
# 5. MODEL 4 — QWEN2.5 3B INSTRUCT LLM VIA OLLAMA
#    ZERO-SHOT + FEW-SHOT
# ============================================================

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"
LLM_MODEL = "qwen2.5:3b-instruct"

# đặt 20 để test nhanh, None để chạy full test
LLM_LIMIT = None

def check_ollama_server_and_model():
    try:
        r = requests.get(OLLAMA_TAGS_URL, timeout=20)
        r.raise_for_status()

        data = r.json()
        models = [m.get("name", "") for m in data.get("models", [])]

        print("Ollama status:", r.status_code)
        print("Available models:")
        for m in models:
            print("-", m)

        if LLM_MODEL not in models:
            raise RuntimeError(
                f"Không tìm thấy model {LLM_MODEL}. "
                f"Hãy chạy: ollama pull {LLM_MODEL}"
            )

        print("\nSelected LLM model:", LLM_MODEL)
        return True

    except Exception as e:
        print("Không gọi được Ollama server hoặc chưa có model cần dùng.")
        print("Hãy mở Ollama app hoặc chạy: ollama serve")
        print("Error:", e)
        return False

server_ok = check_ollama_server_and_model()
if not server_ok:
    raise RuntimeError("Ollama chưa sẵn sàng. Dừng cell tại đây.")


llm_test_df = test_df.copy().reset_index(drop=True)

def get_llm_text(row):
    # LLM dùng text gốc để giữ emoji, slang, dấu câu.
    if "text" in row and not pd.isna(row["text"]) and str(row["text"]).strip() != "":
        return str(row["text"])

    if "model_text" in row and not pd.isna(row["model_text"]) and str(row["model_text"]).strip() != "":
        return str(row["model_text"])

    if "clean_text" in row and not pd.isna(row["clean_text"]) and str(row["clean_text"]).strip() != "":
        return str(row["clean_text"])

    return ""

llm_test_df["llm_text"] = llm_test_df.apply(get_llm_text, axis=1)
llm_test_df = llm_test_df[llm_test_df["llm_text"].str.strip() != ""].copy()

if LLM_LIMIT is not None:
    llm_test_df = llm_test_df.head(LLM_LIMIT).copy()

llm_topics = llm_test_df["topic"].fillna("").astype(str).tolist()
llm_texts = llm_test_df["llm_text"].tolist()
llm_true = llm_test_df["final_manual_label"].astype(int).to_numpy()

print("\nLLM model:", LLM_MODEL)
print("LLM test size:", len(llm_test_df))
print("True label distribution:")
print(pd.Series(llm_true).value_counts().sort_index())


ZERO_SHOT_INSTRUCTION = """
Bạn là hệ thống phân loại cảm xúc comment tiếng Việt trên mạng xã hội.

Phân loại comment thành đúng một trong ba nhãn:
0 = positive: khen, thích, ủng hộ, hài lòng, đồng tình, đánh giá tích cực.
1 = negative: chê, phản đối, thất vọng, mỉa mai, cà khịa, tức giận, đánh giá tiêu cực.
2 = neutral: câu hỏi thông tin, mô tả không cảm xúc, so sánh chưa rõ thái độ, hoặc không đủ ngữ cảnh.

Quy tắc:
- Chỉ trả về duy nhất một ký tự: 0 hoặc 1 hoặc 2.
- Không giải thích.
- Neutral không phải nhãn mặc định.
- Nếu có khen/chê rõ dù comment ngắn, chọn positive hoặc negative.
- Nếu có mỉa mai/cà khịa/châm biếm, chọn negative.
""".strip()


FEW_SHOT_EXAMPLES = """
Ví dụ:

Topic: iphone17series
Comment: ngon quá, bản này đáng mua thật
Label: 0

Topic: iphone17series
Comment: giá này chát quá, không đáng mua
Label: 1

Topic: iphone17series
Comment: khi nào mở bán vậy mọi người
Label: 2

Topic: s25series
Comment: pin yếu vậy thì chán thật
Label: 1

Topic: s25series
Comment: lỗi màn hình vậy ai dám mua
Label: 1

Topic: s25series
Comment: thiết kế năm nay đẹp hơn hẳn
Label: 0

Topic: vinfast_vf3
Comment: xe này giá bao nhiêu vậy
Label: 2

Topic: vinfast_vf3
Comment: nhìn nhỏ mà tiện phết
Label: 0

Topic: vinfast_vf3
Comment: xe gì mà lỗi hoài vậy trời
Label: 1

Topic: arsenal
Comment: đá thế này mới xứng đáng vô địch
Label: 0

Topic: arsenal
Comment: trận này đá như mơ ngủ
Label: 1

Topic: arsenal
Comment: hàng thủ tấu hài quá
Label: 1

Topic: ronaldo
Comment: vẫn là goat, quá đẳng cấp
Label: 0

Topic: ronaldo
Comment: hết thời rồi, đá chán quá
Label: 1

Topic: ronaldo
Comment: năm nay anh ấy ghi bao nhiêu bàn rồi
Label: 2
""".strip()


def build_zero_shot_prompt(topic, comment):
    return f"""{ZERO_SHOT_INSTRUCTION}

Bây giờ hãy phân loại comment sau.

Topic: {topic}
Comment: {comment}
Label:"""


def build_few_shot_prompt(topic, comment):
    return f"""{ZERO_SHOT_INSTRUCTION}

{FEW_SHOT_EXAMPLES}

Bây giờ hãy phân loại comment sau.

Topic: {topic}
Comment: {comment}
Label:"""


def parse_label(output_text):
    if output_text is None:
        return None

    text = str(output_text).strip()

    if text in {"0", "1", "2"}:
        return int(text)

    match = re.search(r"(?:label\s*[:：]?\s*)?([012])", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    return None


def ollama_generate(prompt, timeout=300):
    payload = {
        "model": LLM_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "top_p": 0.9,
            "num_predict": 32,
            "num_ctx": 2048,
            "seed": 42
        }
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    return data.get("response", "").strip()


def predict_one_llm(topic, comment, prompt_type="few-shot", max_retries=2):
    if prompt_type == "zero-shot":
        prompt = build_zero_shot_prompt(topic, comment)
    else:
        prompt = build_few_shot_prompt(topic, comment)

    last_raw = None

    for attempt in range(max_retries + 1):
        try:
            raw = ollama_generate(prompt)
            last_raw = raw
            label = parse_label(raw)

            if label in {0, 1, 2}:
                return label, raw

            strict_prompt = (
                prompt
                + "\n\nBạn đã trả lời sai định dạng. "
                + "Chỉ trả về duy nhất một ký tự 0 hoặc 1 hoặc 2. "
                + "Không giải thích. Label:"
            )

            raw = ollama_generate(strict_prompt)
            last_raw = raw
            label = parse_label(raw)

            if label in {0, 1, 2}:
                return label, raw

        except Exception as e:
            last_raw = f"ERROR: {e}"
            if attempt == max_retries:
                return None, last_raw
            time.sleep(1)

    return None, f"INVALID_OUTPUT | last_raw={last_raw}"


def run_llm_benchmark(prompt_type):
    print(f"\n\n========== LLM QUICK TEST: {prompt_type} ==========")

    quick_preds = []

    for i in range(min(5, len(llm_texts))):
        pred, raw = predict_one_llm(llm_topics[i], llm_texts[i], prompt_type=prompt_type)
        quick_preds.append(pred)

        print("-----")
        print("Topic:", llm_topics[i])
        print("Comment:", llm_texts[i])
        print("True:", llm_true[i])
        print("Pred:", pred)
        print("Raw:", raw)

    if any(p not in {0, 1, 2} for p in quick_preds):
        raise RuntimeError(f"Quick test LLM lỗi ở chế độ {prompt_type}. Dừng để kiểm tra prompt/model.")

    print(f"\nQuick test OK: {prompt_type}")

    preds = []
    raw_outputs = []

    start_time = time.time()

    for topic, comment in tqdm(
        list(zip(llm_topics, llm_texts)),
        total=len(llm_texts),
        desc=f"LLM {prompt_type}"
    ):
        pred, raw = predict_one_llm(topic, comment, prompt_type=prompt_type)
        preds.append(pred)
        raw_outputs.append(raw)

    elapsed = time.time() - start_time

    invalid_count = sum(p not in {0, 1, 2} for p in preds)

    print("\nDone:", prompt_type)
    print("Elapsed seconds:", round(elapsed, 2))
    print("Average seconds/sample:", round(elapsed / len(llm_texts), 3))
    print("Invalid predictions:", invalid_count)

    if invalid_count > 3:
        raise RuntimeError(
            f"Có {invalid_count} prediction lỗi ở chế độ {prompt_type}. "
            "Không được ép hàng loạt về neutral."
        )

    preds_clean = [int(p) if p in {0, 1, 2} else 2 for p in preds]

    print("\nPrediction distribution:")
    print(pd.Series(preds_clean).value_counts().sort_index())

    model_name = f"Qwen2.5 3B Instruct LLM ({prompt_type})"

    evaluate_predictions(
        model_name,
        llm_true,
        preds_clean,
        train_size=0,
        test_size=len(llm_test_df),
        extra_info={
            "best_validation_macro_f1": np.nan,
            "best_checkpoint": f"{prompt_type} via Ollama"
        }
    )

    safe_prompt_type = prompt_type.replace("-", "_")

    pred_df = llm_test_df.copy()
    pred_df[f"pred_llm_{safe_prompt_type}"] = preds_clean
    pred_df[f"pred_llm_{safe_prompt_type}_label_name"] = pd.Series(preds_clean).map(id2label).values
    pred_df[f"llm_raw_output_{safe_prompt_type}"] = raw_outputs

    pred_path = f"before_calibration_predictions_llm_{safe_prompt_type}.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print("\nSaved predictions:", pred_path)

    return preds_clean, raw_outputs


llm_zero_preds, llm_zero_raw = run_llm_benchmark("zero-shot")
llm_few_preds, llm_few_raw = run_llm_benchmark("few-shot")


Ollama status: 200
Available models:
- qwen2.5:3b-instruct
- qwen3.5:4b

Selected LLM model: qwen2.5:3b-instruct

LLM model: qwen2.5:3b-instruct
LLM test size: 281
True label distribution:
0     81
1    106
2     94
Name: count, dtype: int64


========== LLM QUICK TEST: zero-shot ==========
-----
Topic: arsenal
Comment: GÁY TO LÊN ANH EM PHÁO THỦ ƠIII
True: 0
Pred: 1
Raw: 1
-----
Topic: arsenal
Comment: Ai vào đây sau khi Arsenal đã vô địch ?
True: 2
Pred: 2
Raw: 2
-----
Topic: arsenal
Comment: 0:21 0:22 MCT là số 1 arsenal tuổi 1:11 1:11
True: 1
Pred: 2
Raw: 2
-----
Topic: arsenal
Comment: Giờ pháo lại sáng cửa rồi , ngày mai mà thắng WH thì MC gần như hết hi vọng 😅
True: 0
Pred: 1
Raw: 1
-----
Topic: arsenal
Comment: Năm nay mà không vô địch được thì không còn gì để nói. Trách ai bây giờ, trách mình thôi.
True: 1
Pred: 1
Raw: 1

Quick test OK: zero-shot


LLM zero-shot: 100%|██████████| 281/281 [11:13<00:00,  2.40s/it]



Done: zero-shot
Elapsed seconds: 673.32
Average seconds/sample: 2.396
Invalid predictions: 0

Prediction distribution:
0     42
1    110
2    129
Name: count, dtype: int64
========== Qwen2.5 3B Instruct LLM (zero-shot) ==========
Accuracy: 0.5694
Macro-F1: 0.5676
Weighted-F1: 0.5709
Cohen's Kappa: 0.3433

Classification report:
              precision    recall  f1-score   support

    positive     0.8095    0.4198    0.5528        81
    negative     0.6182    0.6415    0.6296       106
     neutral     0.4496    0.6170    0.5202        94

    accuracy                         0.5694       281
   macro avg     0.6258    0.5594    0.5676       281
weighted avg     0.6169    0.5694    0.5709       281


Confusion matrix:
                Pred positive  Pred negative  Pred neutral
Human positive             34             10            37
Human negative              4             68            34
Human neutral               4             32            58

Saved predictions: before_calibr

LLM few-shot: 100%|██████████| 281/281 [11:25<00:00,  2.44s/it]



Done: few-shot
Elapsed seconds: 685.82
Average seconds/sample: 2.441
Invalid predictions: 0

Prediction distribution:
0     79
1     88
2    114
Name: count, dtype: int64
========== Qwen2.5 3B Instruct LLM (few-shot) ==========
Accuracy: 0.6121
Macro-F1: 0.6147
Weighted-F1: 0.6147
Cohen's Kappa: 0.4168

Classification report:
              precision    recall  f1-score   support

    positive     0.6456    0.6296    0.6375        81
    negative     0.7045    0.5849    0.6392       106
     neutral     0.5175    0.6277    0.5673        94

    accuracy                         0.6121       281
   macro avg     0.6226    0.6141    0.6147       281
weighted avg     0.6250    0.6121    0.6147       281


Confusion matrix:
                Pred positive  Pred negative  Pred neutral
Human positive             51              8            22
Human negative             11             62            33
Human neutral              17             18            59

Saved predictions: before_calibrat

In [7]:
# ============================================================
# 6. FINAL BENCHMARK RESULTS
# ============================================================

final_results = pd.DataFrame(benchmark_results).sort_values(
    "macro_f1",
    ascending=False
)

final_results.to_csv(
    "before_calibration_sentiment_benchmark_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== FINAL BENCHMARK RESULTS — BEFORE CALIBRATION ==========")
display(final_results)

print("\nSaved: before_calibration_sentiment_benchmark_results.csv")


========== FINAL BENCHMARK RESULTS — BEFORE CALIBRATION ==========


,model,accuracy,macro_f1,weighted_f1,cohen_kappa,train_size,test_size,best_validation_macro_f1,best_checkpoint
1,PhoBERT-base-v2 Fine-tuned,0.704626,0.709770,0.706314,0.554633,7925,281,0.705388,before_calibration_phobert_sentiment_model\che...
0,TF-IDF + Logistic Regression,0.651246,0.654021,0.650355,0.474636,7925,281,NaN,NaN
4,Qwen2.5 3B Instruct LLM (few-shot),0.612100,0.614661,0.614651,0.416790,0,281,NaN,few-shot via Ollama
2,ViSoBERT Fine-tuned,0.601423,0.602886,0.601451,0.400088,7925,281,0.626640,before_calibration_visobert_sentiment_model\ch...
3,Qwen2.5 3B Instruct LLM (zero-shot),0.569395,0.567552,0.570883,0.343268,0,281,NaN,zero-shot via Ollama



Saved: before_calibration_sentiment_benchmark_results.csv
